### Importing all the files from S1 to S13 into a single DF:

In [1]:
from pathlib import Path
import pandas as pd

# Folder with the CSV files (raw string so the backslashes are read correctly)
data_dir = Path(r"C:\Users\user\_MQ3_Research\Metrics\MQ3_logs_S1-S13")

frames = []
skipped = []

for f in sorted(data_dir.glob("*.csv")):   # only files in this folder, not subfolders
    # Split into at most 5 parts, so yt_ext stays intact even if it contains "_"
    parts = f.stem.split("_", 4)
    if len(parts) != 5:
        skipped.append(f.name)
        continue

    set_id, vd_id, init_soc, user_state, yt_ext = parts

    df = pd.read_csv(f)              # e.g. for filename "S12_V2_95_SQ_G-XZhKqQAHU" 
    df["set_id"] = set_id            # e.g. "S12"
    df["vd_id"] = vd_id              # e.g. "V2"
    df["init_soc"] = int(init_soc)   # e.g. 95 (as a number)
    df["user_state"] = user_state    # e.g. "SQ"
    df["yt_ext"] = yt_ext            # e.g. "G-XZhKqQAHU"
    frames.append(df)

df_all = pd.concat(frames, ignore_index=True)

print(f"Loaded {len(frames)} files, {len(df_all)} rows, {df_all.shape[1]} columns")
if skipped:
    print("Skipped (unexpected file name):", skipped)

df_all.head()

Loaded 70 files, 21707 rows, 46 columns


,id_line,Timestamp,Elapsed (s),Level (%),Voltage (mV),Temp (C),Charge Counter (uAh),Calculated Current (mA),Calculated Power (mW),Accumulated Power (mWh),...,Video Clock,Video Session,Video Position (s),Video Position (mm:ss),Position Source,set_id,vd_id,init_soc,user_state,yt_ext
0,30,2026-10-13 18:31:42,29.61,99,4215,32.0,4953606,1701.79,7173.04,74.763,...,Running,1,0.3,00:00,manual clock,S10,V1,100,ST,KGerjHMa90s
1,31,2026-10-13 18:31:43,30.62,99,4215,32.0,4953606,1701.79,7173.04,77.731,...,Running,1,1.3,00:01,manual clock,S10,V1,100,ST,KGerjHMa90s
2,32,2026-10-13 18:31:45,32.11,99,4215,32.0,4953606,1701.79,7173.04,79.405,...,Running,1,2.8,00:02,manual clock,S10,V1,100,ST,KGerjHMa90s
3,33,2026-10-13 18:31:46,32.95,99,4215,32.0,4953606,1701.79,7173.04,81.617,...,Running,1,3.6,00:03,manual clock,S10,V1,100,ST,KGerjHMa90s
4,34,2026-10-13 18:31:47,34.06,99,4215,32.0,4953606,1701.79,7173.04,83.091,...,Running,1,4.7,00:04,manual clock,S10,V1,100,ST,KGerjHMa90s


### Checking the data is OK...How many Sets? Unique Videos?

In [2]:
meta_cols = ["set_id", "vd_id", "init_soc", "user_state", "yt_ext"]

for col in meta_cols:
    values = df_all[col].unique()
    # Sort set_id / vd_id by their number (S2 before S10), others normally
    if col in ("set_id", "vd_id"):
        values = sorted(values, key=lambda v: int(v[1:]))
    else:
        values = sorted(values)
    print(f"{col} ({len(values)} unique): {list(values)}\n")

set_id (13 unique): ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 'S13']

vd_id (6 unique): ['V1', 'V2', 'V3', 'V4', 'V5', 'V6']

init_soc (7 unique): [np.int64(50), np.int64(75), np.int64(80), np.int64(85), np.int64(90), np.int64(95), np.int64(100)]

user_state (3 unique): ['DY', 'SQ', 'ST']

yt_ext (6 unique): ['G-XZhKqQAHU', 'HI7mTIxNotQ', 'KGerjHMa90s', 'Zgaw2eNP9eo', 'dwHBpykTloY', 'nF8UTFHpmjE']



### Listing all the expriments (Sets/Videos)

In [3]:
import pandas as pd

def to_mmss(seconds):
    """Convert seconds to an 'mm:ss' string (rounded to the nearest second)."""
    if pd.isna(seconds):
        return None
    total = int(round(seconds))
    minutes, secs = divmod(total, 60)
    return f"{minutes:02d}:{secs:02d}"

# Sets to leave out of the analysis
exclude_sets = ["S5", "S6", "S8", "S9"]

# Max video position (in seconds) for each video and set
max_pos = (
    df_all[~df_all["set_id"].isin(exclude_sets)]
          .groupby(["yt_ext", "set_id"])["Video Position (s)"]
          .max()
          .reset_index()
)

# Pivot: one row per yt_ext, one column per set_id
pivot = max_pos.pivot(index="yt_ext", columns="set_id", values="Video Position (s)")

# Order set columns numerically (S2 before S10)
pivot = pivot[sorted(pivot.columns, key=lambda s: int(s[1:]))]

# Convert seconds to mm:ss (empty cell = that video wasn't played in that set)
pivot_mmss = pivot.apply(lambda col: col.map(to_mmss)).fillna("")

print(f"{pivot_mmss.shape[0]} unique videos x {pivot_mmss.shape[1]} sets")
pivot_mmss

6 unique videos x 9 sets


set_id,S1,S2,S3,S4,S7,S10,S11,S12,S13
yt_ext,,,,,,,,,
G-XZhKqQAHU,04:54,04:55,04:54,04:56,04:53,05:36,04:56,04:56,04:56
HI7mTIxNotQ,06:14,06:14,06:14,06:14,06:14,06:17,06:18,06:15,06:13
KGerjHMa90s,06:50,06:12,06:10,06:16,06:12,06:19,06:18,06:12,06:15
Zgaw2eNP9eo,03:36,03:37,03:38,03:41,03:37,03:37,03:40,03:39,03:38
dwHBpykTloY,04:33,04:29,04:35,04:35,04:34,04:34,04:35,04:36,04:34
nF8UTFHpmjE,06:41,06:40,06:40,06:15,06:45,06:40,06:41,06:50,06:40


In [4]:
# Sets to leave out of the analysis
exclude_sets = ["S5", "S6", "S8", "S9"]

max_power = (
    df_all[~df_all["set_id"].isin(exclude_sets)]
          .groupby(["yt_ext", "set_id"])["Accumulated Power (mWh)"]
          .max()
          .reset_index()
)

# Pivot: one row per yt_ext, one column per set_id
pivot_power = max_power.pivot(index="yt_ext", columns="set_id", values="Accumulated Power (mWh)")

# Order set columns numerically (S2 before S10)
pivot_power = pivot_power[sorted(pivot_power.columns, key=lambda s: int(s[1:]))]

print(f"{pivot_power.shape[0]} unique videos x {pivot_power.shape[1]} sets")
pivot_power.round(2)

6 unique videos x 9 sets


set_id,S1,S2,S3,S4,S7,S10,S11,S12,S13
yt_ext,,,,,,,,,
G-XZhKqQAHU,746.04,959.78,792.04,991.48,630.04,695.45,712.26,1715.69,754.73
HI7mTIxNotQ,1011.01,1087.55,913.20,922.88,924.58,868.55,920.46,4538.62,890.14
KGerjHMa90s,1023.04,1102.73,1036.43,1059.56,1013.47,851.11,872.55,1009.01,933.90
Zgaw2eNP9eo,548.96,606.61,635.86,341.37,514.12,519.22,557.14,3564.29,591.42
dwHBpykTloY,716.63,870.89,664.11,736.05,616.09,591.28,613.01,5198.82,636.71
nF8UTFHpmjE,1071.05,1420.98,1254.62,1474.49,1068.65,1007.66,1096.31,2922.33,1213.13
